In [3]:
# Step 1: Install Dependencies

!pip install chromadb
!pip install sentence-transformers
!pip install pandas
!pip install transformers
!pip install accelerate

print("All dependencies Installed")

All dependencies Installed


In [2]:
# Step 2: Import Dependencies

import pandas as pd
import chromadb

from sentence_transformers import SentenceTransformer

print("Dependecies Imported Successfully!")

Dependecies Imported Successfully!


In [7]:
df = pd.read_csv("careers_dataset.csv")
print(df.head())

                   Career                   Domain              Skills  \
0             AI Engineer  Artificial Intelligence      Python, ML, DL   
1          Data Scientist             Data Science  Python, Statistics   
2  Cyber Security Analyst                 Security    Network Security   
3          Cloud Engineer                    Cloud         AWS, Docker   
4         DevOps Engineer                   DevOps   CI/CD, Kubernetes   

            Certifications                   Description  
0     TensorFlow Developer    Builds intelligent systems  
1         IBM Data Science       Analyzes large datasets  
2                      CEH      Protects digital systems  
3  AWS Solutions Architect  Manages cloud infrastructure  
4         Docker Associate         Automates deployments  


In [8]:
# Step 3: Creating Embeddings

model = SentenceTransformer(
    'all-MiniLM-L6-v2'
)

embeddings = model.encode(
    df["Description"].tolist()
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [9]:
# Step 4: Creating ChromaDB Collection

client = chromadb.Client()

collection = client.create_collection(
    name="career_guidance"
)

In [11]:
# Step 5: Semantic Search

query = """
I enjoy building AI systems
and creating chatbots.
"""

query_embedding = model.encode([query])

from sklearn.metrics.pairwise import cosine_similarity

cosine_similarity(
    query_embedding,
    embeddings
)

array([[0.47377557, 0.1321373 , 0.13399035, 0.09021994, 0.04717561,
        0.19966853, 0.1690548 , 0.23733935, 0.5045392 , 0.5497472 ]],
      dtype=float32)

## Chapter 6: Semantic Search Results


In [15]:
import numpy as np

similarity_scores = cosine_similarity(query_embedding, embeddings).flatten()

df['similarity_score'] = similarity_scores
top_recommendations = df.sort_values(by='similarity_score', ascending=False)

print("Top Career Recommendations based on Semantic Search:")
display(top_recommendations[['Career', 'Description', 'similarity_score']].head(3))

Top Career Recommendations based on Semantic Search:


,Career,Description,similarity_score
9,AI Agent Developer,Builds intelligent AI agents,0.549747
8,MLOps Engineer,Deploys AI systems,0.504539
0,AI Engineer,Builds intelligent systems,0.473776


## Chapter 7: RAG Pipeline


In [16]:


collection.add(
    ids=[str(i) for i in range(len(df))],
    embeddings=embeddings.tolist(),
    metadatas=df[['Career', 'Domain']].to_dict('records'),
    documents=df['Description'].tolist()
)

rag_results = collection.query(
    query_embeddings=query_embedding.tolist(),
    n_results=3
)

print("RAG Pipeline Results from ChromaDB:")
for i in range(len(rag_results['ids'][0])):
    print(f"ID: {rag_results['ids'][0][i]}, Career: {rag_results['metadatas'][0][i]['Career']}, Distance: {rag_results['distances'][0][i]}")

RAG Pipeline Results from ChromaDB:
ID: 9, Career: AI Agent Developer, Distance: 0.9005057215690613
ID: 8, Career: MLOps Engineer, Distance: 0.9909216165542603
ID: 0, Career: AI Engineer, Distance: 1.0524488687515259


## Chapter 8: Results and Evaluation


In [17]:
print(f"User Query: {query}")
print("-" * 30)
print(f"Semantic Search Best Match: {top_recommendations.iloc[0]['Career']}")
print(f"RAG Pipeline Best Match: {rag_results['metadatas'][0][0]['Career']}")

if top_recommendations.iloc[0]['Career'] == rag_results['metadatas'][0][0]['Career']:
    print("\nEvaluation: Both systems are consistent and provided the same top recommendation.")
else:
    print("\nEvaluation: There is a discrepancy between the systems, likely due to distance metric differences (Cosine vs L2).")

User Query: 
I enjoy building AI systems
and creating chatbots.

------------------------------
Semantic Search Best Match: AI Agent Developer
RAG Pipeline Best Match: AI Agent Developer

Evaluation: Both systems are consistent and provided the same top recommendation.
